In [9]:
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages

ROOT = Path("/Users/andreali/Documents/Subgraph_Federated_Learning/")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from andrea.multigraph_generation import TASKS
from andrea.run_fedavg import SELECT_SUBSET_PATH, EXPERIMENT_LOG_FOLDER, SELECT_SUBSET

from andrea.heterogeneity.pairwise_selection import build_family_specs

SELECT_SUBSET_PATH = "heterogeneity"
SELECT_SUBSET = "selected_pairs"

EXPERIMENT_LOG_FOLDER = "pairwise_selection_experiment"
DATA_DIR = "test_generation_parameters.csv"

SELECTED_SUBSETS_CSV_PATH = Path(f"./{SELECT_SUBSET_PATH}/{SELECT_SUBSET}.csv")
EXP_LOG_EPOCH1_PATH = Path(f"./{EXPERIMENT_LOG_FOLDER}/experiment_log_epoch1.csv")
EXP_LOG_EPOCH3_PATH = Path(f"./{EXPERIMENT_LOG_FOLDER}/experiment_log_epoch3.csv")
EXP_LOG_EPOCH5_PATH = Path(f"./{EXPERIMENT_LOG_FOLDER}/experiment_log_epoch5.csv")
EXP_LOG_EPOCH10_PATH = Path(f"./{EXPERIMENT_LOG_FOLDER}/experiment_log_epoch10.csv")
TEST_GEN_PATH = ROOT / Path("./andrea/test_generation_parameters.csv")

selected_pairs = pd.read_csv(SELECTED_SUBSETS_CSV_PATH)
print("Loaded selected pairs rows:", len(selected_pairs))
exp_log_epoch1 = pd.read_csv(EXP_LOG_EPOCH1_PATH)
exp_log_epoch3 = pd.read_csv(EXP_LOG_EPOCH3_PATH)
exp_log_epoch5 = pd.read_csv(EXP_LOG_EPOCH5_PATH)
exp_log_epoch10 = pd.read_csv(EXP_LOG_EPOCH10_PATH)

print("epoch1 rows:", len(exp_log_epoch1))
print("epoch3 rows:", len(exp_log_epoch3))
print("epoch5 rows:", len(exp_log_epoch5))
print("epoch10 rows:", len(exp_log_epoch10))

test_gen = pd.read_csv(TEST_GEN_PATH)
print("Loaded generation rows:", len(test_gen))

Loaded selected pairs rows: 4
epoch1 rows: 126
epoch3 rows: 63
epoch5 rows: 45
epoch10 rows: 45
Loaded generation rows: 880


## General helpers

In [10]:
# ============================================================
# GENERAL HELPERS
# ============================================================


def resolve_existing_path(path_like):
    p = Path(path_like)
    candidates = [
        p,
        Path.cwd() / p,
        ROOT / p,
    ]
    for cand in candidates:
        if cand.exists():
            return cand.resolve()
    raise FileNotFoundError(f"Could not resolve path: {path_like}")


def load_run_csv(csv_path_like) -> pd.DataFrame:
    csv_path = resolve_existing_path(csv_path_like)
    return pd.read_csv(csv_path)


def get_task_split_stat(graph_row, split_name, task, kind):
    col = f"{split_name}_{task}_{kind}"
    if graph_row is None:
        return None
    if col in graph_row.index:
        return graph_row[col]
    return None


def build_family_spec_map():
    return {spec.name: spec for spec in build_family_specs()}


PRETTY_METRIC_NAMES = {
    "shared_support_frac": "shared_support",
    "task_profile_gap": "task_profile_gap",
    "homophily_gap": "homophily_gap",
    "structure_gap": "structure_gap",
    "size_mismatch": "size_mismatch",
    "support_regime_gap": "support_regime_gap",
    "low_support_task_frac": "low_support_task_frac",
    "complementarity_score": "complementarity",
    "incompatibility_score": "incompatibility",
}


def _fmt_val(x, nd=3):
    if x is None or pd.isna(x):
        return "NA"
    return f"{float(x):.{nd}f}"


def _format_metric_list(metric_names, pair_row):
    items = []
    for m in metric_names:
        label = PRETTY_METRIC_NAMES.get(m, m)
        value = pair_row[m] if m in pair_row.index else np.nan
        items.append(f"{label}={_fmt_val(value)}")
    return "[" + ", ".join(items) + "]"


def build_pair_family_footer(pair_row):
    if pair_row is None:
        return "family=NA"

    family = str(pair_row.get("family", "unknown"))
    spec = build_family_spec_map().get(family)

    if spec is None:
        return f"family={family}"

    line1 = (
        f"family={family} | "
        f"selection_score={_fmt_val(pair_row.get('selection_score'))}"
    )

    max_part = (
        "maximized=" + _format_metric_list(list(spec.maximize.keys()), pair_row)
        if spec.maximize
        else "maximized=[]"
    )

    min_part = (
        "minimized=" + _format_metric_list(list(spec.minimize.keys()), pair_row)
        if spec.minimize
        else "minimized=[]"
    )

    line2 = f"{max_part} | {min_part}"
    return line1 + "\n" + line2

## Subset helpers

In [11]:
PAIR_META_COLS = [
    "family",
    "rank_within_family",
    "selection_score",
    "subset_id",
    "subset_size",
    "graph_id_i",
    "graph_id_j",
    "dataset_i",
    "dataset_j",
    "type_i",
    "type_j",
    "shared_support_frac",
    "support_regime_gap",
    "low_support_task_frac",
    "task_profile_gap",
    "structure_gap",
    "homophily_gap",
    "size_mismatch",
    "complementarity_score",
    "incompatibility_score",
    "label_top_gap_task",
    "label_top_gap_value",
    "motif_top_gap_task",
    "motif_top_gap_value",
    "task_homophily_top_gap_task",
    "task_homophily_top_gap_value",
    "label_gap_json",
    "label_rates_i_json",
    "label_rates_j_json",
    "label_counts_i_json",
    "label_counts_j_json",
    "num_nodes_i",
    "num_nodes_j",
    "num_edges_i",
    "num_edges_j",
    "homophily_i",
    "homophily_j",
    "taskwise_homophily_i_json",
    "taskwise_homophily_j_json",
    "task_homophily_gap_json",
]

def parse_subset_clients_str(x):
    return [int(x) for x in str(x).split("|") if str(x).strip() != ""]

def build_pair_run_table(selected_pairs_df, exp_log_df):
    out = []

    for _, pair_row in selected_pairs_df.iterrows():
        subset_clients = str(pair_row["subset_clients"])
        client_ids = parse_subset_clients_str(subset_clients)

        subset_logs = exp_log_df[
            exp_log_df["subset_clients"].astype(str) == subset_clients
        ].copy()
        if subset_logs.empty:
            continue

        fed_rows = subset_logs[subset_logs["run_type"] == "fedavg"].copy()
        local_rows = subset_logs[subset_logs["run_type"] == "local"].copy()
        if fed_rows.empty or local_rows.empty:
            continue

        for model_tag, fed_grp in fed_rows.groupby("model_tag", sort=False):
            for client_id in client_ids:
                local_client_rows = local_rows[
                    local_rows["graph_id"].astype(str) == str(client_id)
                ].copy()

                matched = []
                for _, fed_row in fed_grp.sort_values("seed").iterrows():
                    seed = int(fed_row["seed"])

                    cand = local_client_rows[
                        (local_client_rows["seed"] == seed)
                        & (local_client_rows["model_tag"] == model_tag)
                    ].copy()

                    if cand.empty:
                        continue

                    local_row = cand.iloc[-1]
                    matched.append(
                        {
                            "seed": seed,
                            "local_csv": resolve_existing_path(local_row["out_csv"]),
                            "fedavg_csv": resolve_existing_path(fed_row["out_csv"]),
                            "dataset_id": local_row["dataset_id"],
                            "rounds": fed_row["rounds"],
                            "local_epochs": fed_row["local_epochs"],
                            "selection_metric": fed_row["selection_metric"],
                        }
                    )

                if not matched:
                    continue

                matched = sorted(matched, key=lambda x: x["seed"])

                item = {
                    "subset_id": pair_row.get("subset_id"),
                    "subset_clients": subset_clients,
                    "family": pair_row.get("family"),
                    "graph_id": str(client_id),
                    "dataset_id": matched[0]["dataset_id"],
                    "model_tag": model_tag,
                    "rounds": matched[0]["rounds"],
                    "local_epochs": matched[0]["local_epochs"],
                    "selection_metric": matched[0]["selection_metric"],
                    "seeds": [m["seed"] for m in matched],
                    "n_seeds": len(matched),
                    "local_csvs": [m["local_csv"] for m in matched],
                    "fedavg_csvs": [m["fedavg_csv"] for m in matched],
                }

                for c in PAIR_META_COLS:
                    if c in pair_row.index:
                        item[c] = pair_row[c]

                out.append(item)

    out_df = pd.DataFrame(out)
    if not out_df.empty:
        out_df = out_df.drop_duplicates(
            subset=["subset_clients", "graph_id", "model_tag"],
            keep="last",
        ).reset_index(drop=True)

    return out_df

## Data collection for plotting


In [12]:
def aggregate_seed_curves(curves, value_col):
    parts = []
    for seed_idx, curve in enumerate(curves):
        if curve is None or curve.empty:
            continue
        part = curve[["step", value_col]].dropna().copy()
        if part.empty:
            continue
        part["seed_idx"] = seed_idx
        parts.append(part)

    if not parts:
        return pd.DataFrame(columns=["step", "mean", "std", "count"])

    full = pd.concat(parts, axis=0, ignore_index=True)
    agg = (
        full.groupby("step")[value_col]
        .agg(["mean", "std", "count"])
        .reset_index()
        .sort_values("step")
    )
    agg["std"] = agg["std"].fillna(0.0)
    return agg


def get_scalar_seed_stats(
    dfs,
    *,
    phase,
    split,
    metric_col,
    graph_id=None,
    local_epochs=1,
):
    curves = []
    for df in dfs:
        part = get_scalar_curve(
            df,
            phase=phase,
            split=split,
            metric_col=metric_col,
            graph_id=graph_id,
            local_epochs=local_epochs,
        )
        curves.append(part)
    return aggregate_seed_curves(curves, metric_col)


def get_task_seed_stats(
    dfs,
    *,
    phase,
    split,
    task,
    metric_col,
    graph_id=None,
    local_epochs=1,
):
    curves = []
    for df in dfs:
        part = get_task_curve(
            df,
            phase=phase,
            split=split,
            task=task,
            metric_col=metric_col,
            graph_id=graph_id,
            local_epochs=local_epochs,
        )
        curves.append(part)
    return aggregate_seed_curves(curves, metric_col)


def get_delta_seed_stats(
    left_dfs,
    right_dfs,
    *,
    phase_left,
    phase_right,
    split,
    metric_col,
    graph_id_right=None,
    task=None,
    local_epochs=1,
):
    delta_curves = []

    for left_df, right_df in zip(left_dfs, right_dfs):
        if task is None:
            left = get_scalar_curve(
                left_df,
                phase=phase_left,
                split=split,
                metric_col=metric_col,
                local_epochs=local_epochs,
            )
            right = get_scalar_curve(
                right_df,
                phase=phase_right,
                split=split,
                metric_col=metric_col,
                graph_id=graph_id_right,
                local_epochs=local_epochs,
            )
        else:
            left = get_task_curve(
                left_df,
                phase=phase_left,
                split=split,
                task=task,
                metric_col=metric_col,
                local_epochs=local_epochs,
            )
            right = get_task_curve(
                right_df,
                phase=phase_right,
                split=split,
                task=task,
                metric_col=metric_col,
                graph_id=graph_id_right,
                local_epochs=local_epochs,
            )

        if left.empty or right.empty:
            continue

        merged = pd.merge(
            left[["step", metric_col]],
            right[["step", metric_col]],
            on="step",
            how="inner",
            suffixes=("_left", "_right"),
        )
        if merged.empty:
            continue

        merged["delta"] = merged[f"{metric_col}_left"] - merged[f"{metric_col}_right"]
        delta_curves.append(merged[["step", "delta"]])

    return aggregate_seed_curves(delta_curves, "delta")


TASK_COLORS = {
    "cycle2": "#1f77b4",
    "cycle3": "#ff7f0e",
    "cycle4": "#2ca02c",
    "cycle5": "#d62728",
    "cycle6": "#9467bd",
}


def plot_mean_std(
    ax,
    agg_df,
    *,
    label,
    linestyle="-",
    color=None,
    alpha_fill=0.16,
    linewidth=2.0,
):
    if agg_df is None or agg_df.empty:
        return False

    x = agg_df["step"].to_numpy()
    y = agg_df["mean"].to_numpy()
    s = agg_df["std"].to_numpy()

    (line,) = ax.plot(
        x,
        y,
        label=label,
        linestyle=linestyle,
        color=color,
        linewidth=linewidth,
    )
    fill_color = line.get_color()
    ax.fill_between(x, y - s, y + s, alpha=alpha_fill, color=fill_color)
    return True


def _effective_step(df, local_epochs):
    df = df.copy()
    if df.empty:
        return df

    phase = None
    if "phase" in df.columns and df["phase"].notna().any():
        phase = str(df["phase"].dropna().iloc[0])

    has_round = "round" in df.columns and df["round"].notna().any()
    has_local_epoch = "local_epoch" in df.columns and df["local_epoch"].notna().any()

    ROUND_BASED_PHASES = {
        "val_epoch",  # fedavg pre-aggregation evals
        "val_epoch_task",  # fedavg pre-aggregation per-task evals
        "global_val_client",
        "global_val_client_task",
        "global_val_mean",
        "best_global_train",
        "best_global_train_task",
        "best_global_val",
        "best_global_val_task",
        "best_global_test",
        "best_global_test_task",
    }

    # FedAvg eval phases: one point per round
    if phase in ROUND_BASED_PHASES and has_round:
        df["step"] = df["round"].astype(int)

    # phases that truly have both round and local_epoch (e.g. train_epoch in fedavg)
    elif has_round and has_local_epoch:
        df["step"] = (df["round"].astype(int) - 1) * int(local_epochs) + df[
            "local_epoch"
        ].astype(int)

    # local standalone phases
    elif has_local_epoch:
        df["step"] = df["local_epoch"].astype(int)

    # fallback
    elif has_round:
        df["step"] = df["round"].astype(int)

    else:
        raise ValueError("Could not infer step from dataframe")

    return df


def get_scalar_curve(df, *, phase, split, metric_col, graph_id=None, local_epochs=1):
    part = df[
        (df["phase"] == phase) & (df["split"] == split) & (df["task"].isna())
    ].copy()
    if graph_id is not None:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        return part

    part = _effective_step(part, local_epochs)
    return part[["step", metric_col]].dropna().sort_values("step")


def get_task_curve(
    df, *, phase, split, task, metric_col, graph_id=None, local_epochs=1
):
    part = df[
        (df["phase"] == phase) & (df["split"] == split) & (df["task"] == task)
    ].copy()

    if graph_id is not None:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        return part

    part = _effective_step(part, local_epochs)
    return part[["step", metric_col]].dropna().sort_values("step")


def fmt_percent(rate):
    if rate is None or pd.isna(rate):
        return "NA"
    return f"{100.0 * float(rate):.1f}%"


def fmt_count(x):
    if x is None or pd.isna(x):
        return "NA"
    return f"{int(round(float(x)))}"


def fmt_rate_count(rate, count):
    return f"rate={fmt_percent(rate)}, count={fmt_count(count)}"


def _soft_cell_color(value, *, cmap_name="Blues", vmin=0.0, vmax=0.7, alpha=0.35):
    """
    value should be in raw rate space, e.g. 0.249 not 24.9
    """
    if value is None or pd.isna(value):
        return (1, 1, 1, 1)

    value = float(value)
    norm = (value - vmin) / max(vmax - vmin, 1e-12)
    norm = min(max(norm, 0.0), 1.0)

    cmap = plt.get_cmap(cmap_name)
    rgba = cmap(0.18 + 0.55 * norm)
    return (rgba[0], rgba[1], rgba[2], alpha)


def get_task_count_curve(
    df,
    *,
    phase,
    split,
    task,
    graph_id=None,
    local_epochs=1,
):
    part = df[
        (df["phase"] == phase) & (df["split"] == split) & (df["task"] == task)
    ].copy()

    if graph_id is not None:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()

    needed = ["tp", "fp", "tn", "fn"]
    if part.empty or any(c not in part.columns for c in needed):
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    part = _effective_step(part, local_epochs)
    part = part[["step", "tp", "fp", "tn", "fn"]].dropna()
    return part.sort_values("step").reset_index(drop=True)


def binary_f1_from_counts(tp, fp, fn):
    denom = 2.0 * tp + fp + fn
    out = np.full_like(tp, np.nan, dtype=float)
    valid = denom > 0
    out[valid] = (2.0 * tp[valid]) / denom[valid]
    return out


def get_pair_pooled_task_f1_seed_stats(
    dfs_left,
    dfs_right,
    *,
    phase,
    split,
    task,
    graph_id_left=None,
    graph_id_right=None,
    local_epochs=1,
):
    curves = []

    for df_left, df_right in zip(dfs_left, dfs_right):
        left = get_task_count_curve(
            df_left,
            phase=phase,
            split=split,
            task=task,
            graph_id=graph_id_left,
            local_epochs=local_epochs,
        )
        right = get_task_count_curve(
            df_right,
            phase=phase,
            split=split,
            task=task,
            graph_id=graph_id_right,
            local_epochs=local_epochs,
        )

        if left.empty or right.empty:
            continue

        merged = pd.merge(
            left,
            right,
            on="step",
            how="inner",
            suffixes=("_l", "_r"),
        )
        if merged.empty:
            continue

        tp = merged["tp_l"].to_numpy(dtype=float) + merged["tp_r"].to_numpy(dtype=float)
        fp = merged["fp_l"].to_numpy(dtype=float) + merged["fp_r"].to_numpy(dtype=float)
        fn = merged["fn_l"].to_numpy(dtype=float) + merged["fn_r"].to_numpy(dtype=float)

        pooled_f1 = binary_f1_from_counts(tp, fp, fn)

        curve = pd.DataFrame(
            {
                "step": merged["step"].to_numpy(),
                "pooled_f1": pooled_f1,
            }
        ).dropna()

        if not curve.empty:
            curves.append(curve)

    return aggregate_seed_curves(curves, "pooled_f1")


def parse_json_vec_safe(x):
    if x is None or pd.isna(x):
        return np.array([], dtype=float)
    if isinstance(x, str):
        return np.asarray(json.loads(x), dtype=float)
    return np.asarray(x, dtype=float)


def dominant_task_and_rate(rate_vec):
    rate_vec = np.asarray(rate_vec, dtype=float)
    if rate_vec.size == 0:
        return "NA", np.nan
    idx = int(np.argmax(rate_vec))
    return TASKS[idx], float(rate_vec[idx])


def weakest_task_and_rate(rate_vec):
    rate_vec = np.asarray(rate_vec, dtype=float)
    if rate_vec.size == 0:
        return "NA", np.nan
    idx = int(np.argmin(rate_vec))
    return TASKS[idx], float(rate_vec[idx])


def rate_spread(rate_vec):
    rate_vec = np.asarray(rate_vec, dtype=float)
    if rate_vec.size == 0:
        return np.nan
    return float(rate_vec.max() - rate_vec.min())


def get_label_gap_row(pair_row, task_name):
    rows = json.loads(pair_row["label_gap_json"])
    for r in rows:
        if str(r["task"]) == str(task_name):
            return r
    return None


def fmt_rate_pct(x):
    if x is None or pd.isna(x):
        return "NA"
    return f"{100 * float(x):.1f}%"


def fmt_float3(x):
    if x is None or pd.isna(x):
        return "NA"
    return f"{float(x):.3f}"


def safe_int_str(x):
    if x is None or pd.isna(x):
        return "NA"
    return str(int(float(x)))


def family_metric_rows(pair_row):
    family = str(pair_row["family"])

    rates_i = parse_json_vec_safe(pair_row.get("label_rates_i_json"))
    rates_j = parse_json_vec_safe(pair_row.get("label_rates_j_json"))
    th_i = parse_json_vec_safe(pair_row.get("taskwise_homophily_i_json"))
    th_j = parse_json_vec_safe(pair_row.get("taskwise_homophily_j_json"))

    dom_i, dom_rate_i = dominant_task_and_rate(rates_i)
    dom_j, dom_rate_j = dominant_task_and_rate(rates_j)

    weak_i, weak_rate_i = weakest_task_and_rate(rates_i)
    weak_j, weak_rate_j = weakest_task_and_rate(rates_j)

    rows = []

    if "homophily_i" in pair_row.index and "homophily_j" in pair_row.index:
        rows.append(
            (
                "homophily_jaccard",
                fmt_float3(pair_row.get("homophily_i")),
                f"gap={fmt_float3(pair_row.get('homophily_gap'))}",
                fmt_float3(pair_row.get("homophily_j")),
            )
        )

    rows.append(
        (
            "dominant task",
            f"{dom_i} ({fmt_rate_pct(dom_rate_i)})",
            f"task_profile_gap={fmt_float3(pair_row.get('task_profile_gap'))}",
            f"{dom_j} ({fmt_rate_pct(dom_rate_j)})",
        )
    )

    rows.append(
        (
            "weakest task",
            f"{weak_i} ({fmt_rate_pct(weak_rate_i)})",
            "",
            f"{weak_j} ({fmt_rate_pct(weak_rate_j)})",
        )
    )

    rows.append(
        (
            "rate spread",
            fmt_rate_pct(rate_spread(rates_i)),
            "",
            fmt_rate_pct(rate_spread(rates_j)),
        )
    )

    if family in {"task_profile_only", "complementary", "incompatible", "similar"}:
        top_task = pair_row.get("label_top_gap_task", None)
        gap_row = (
            get_label_gap_row(pair_row, top_task) if top_task is not None else None
        )
        if gap_row is not None:
            rows.append(
                (
                    f"label top-gap ({top_task})",
                    fmt_rate_pct(gap_row["value_i"]),
                    f"gap={fmt_rate_pct(gap_row['gap'])}",
                    fmt_rate_pct(gap_row["value_j"]),
                )
            )

    if family in {"homophily_only", "incompatible"} and th_i.size > 0 and th_j.size > 0:
        rows.append(
            (
                "mean task-homophily",
                fmt_float3(np.mean(th_i)),
                f"gap={fmt_float3(pair_row.get('task_homophily_mean_abs_gap'))}",
                fmt_float3(np.mean(th_j)),
            )
        )

        gaps = np.abs(th_i - th_j)
        idx = int(np.argmax(gaps))
        rows.append(
            (
                f"task-h top-gap ({TASKS[idx]})",
                fmt_float3(th_i[idx]),
                f"gap={fmt_float3(gaps[idx])}",
                fmt_float3(th_j[idx]),
            )
        )

    if family in {"complementary", "similar", "incompatible"}:
        rows.append(
            (
                "shared support",
                "—",
                fmt_rate_pct(pair_row.get("shared_support_frac")),
                "—",
            )
        )

    if family in {"incompatible"}:
        rows.append(
            (
                "num_nodes",
                safe_int_str(pair_row.get("num_nodes_i")),
                f"size_mismatch={fmt_float3(pair_row.get('size_mismatch'))}",
                safe_int_str(pair_row.get("num_nodes_j")),
            )
        )
        rows.append(
            (
                "num_edges",
                safe_int_str(pair_row.get("num_edges_i")),
                f"struct_gap={fmt_float3(pair_row.get('structure_gap'))}",
                safe_int_str(pair_row.get("num_edges_j")),
            )
        )

    return rows

# Extra pair plot

In [13]:
def get_graph_meta_row(graph_meta_df, graph_id):
    hit = graph_meta_df[graph_meta_df["graph_id"].astype(str) == str(graph_id)]
    if hit.empty:
        return None
    return hit.iloc[0]


def draw_pair_train_stats_table(
    ax,
    *,
    graph_id_left,
    graph_id_right,
    graph_meta_df,
    task_names,
    fontsize=13,
):
    row_left = get_graph_meta_row(graph_meta_df, graph_id_left)
    row_right = get_graph_meta_row(graph_meta_df, graph_id_right)

    ax.axis("off")
    ax.set_anchor("N")

    if row_left is None or row_right is None:
        ax.text(
            0.5, 0.5,
            "Missing graph metadata for one or both graphs",
            ha="center", va="center", fontsize=fontsize
        )
        return

    row_labels = []
    cell_text = []
    left_rates = []
    right_rates = []
    gap_rates = []

    for task in task_names:
        r_left = get_task_split_stat(row_left, "train", task, "pos_rate")
        n_left = get_task_split_stat(row_left, "train", task, "pos_nodes")

        r_right = get_task_split_stat(row_right, "train", task, "pos_rate")
        n_right = get_task_split_stat(row_right, "train", task, "pos_nodes")

        delta_r = np.nan
        if not pd.isna(r_left) and not pd.isna(r_right):
            delta_r = abs(float(r_left) - float(r_right))

        row_labels.append(task)
        left_rates.append(r_left)
        right_rates.append(r_right)
        gap_rates.append(delta_r)

        cell_text.append([
            fmt_rate_count(r_left, n_left),
            f"gap={fmt_percent(delta_r)}" if not pd.isna(delta_r) else "gap=NA",
            fmt_rate_count(r_right, n_right),
        ])

    table = ax.table(
        cellText=cell_text,
        rowLabels=row_labels,
        colLabels=[f"graph {graph_id_left}", "gap", f"graph {graph_id_right}"],
        loc="center",
        cellLoc="center",
        rowLoc="center",
        bbox=[0, 0, 1, 1],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1.03, 1.20)

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("0.75")
        if r == 0 or c == -1:
            cell.set_text_props(weight="bold")
        if r == 0:
            cell.set_facecolor("#f2f2f2")

    for i in range(len(task_names)):
        r = i + 1
        table[(r, 0)].set_facecolor(
            _soft_cell_color(left_rates[i], cmap_name="Blues", vmin=0.0, vmax=0.7, alpha=0.35)
        )
        table[(r, 1)].set_facecolor(
            _soft_cell_color(gap_rates[i], cmap_name="Reds", vmin=0.0, vmax=0.7, alpha=0.35)
        )
        table[(r, 2)].set_facecolor(
            _soft_cell_color(right_rates[i], cmap_name="Greens", vmin=0.0, vmax=0.7, alpha=0.35)
        )
                
def make_pair_overview_figure(pair_group_df, graph_meta_df, save_root=None):
    if len(pair_group_df) != 2:
        raise ValueError("pair_group_df must contain exactly 2 rows.")

    pair_group_df = pair_group_df.sort_values("graph_id").reset_index(drop=True)
    row_left = pair_group_df.iloc[0]
    row_right = pair_group_df.iloc[1]

    graph_id_left = str(row_left["graph_id"])
    graph_id_right = str(row_right["graph_id"])
    subset_clients = str(row_left["subset_clients"])
    family = str(row_left["family"])
    model_tag = str(row_left["model_tag"])
    local_epochs = int(row_left["local_epochs"])
    n_seeds = int(row_left["n_seeds"])

    local_dfs_left = [load_run_csv(p) for p in row_left["local_csvs"]]
    fed_dfs_left   = [load_run_csv(p) for p in row_left["fedavg_csvs"]]

    local_dfs_right = [load_run_csv(p) for p in row_right["local_csvs"]]
    fed_dfs_right   = [load_run_csv(p) for p in row_right["fedavg_csvs"]]

    fig = plt.figure(figsize=(22, 27), constrained_layout=False)

    gs = GridSpec(
        nrows=7, ncols=2, figure=fig,
        height_ratios=[1.90, 1.25, 1.25, 1.25, 1.25, 1.25, 1.25],
        hspace=0.45, wspace=0.24
    )

    # top tables
    top_gs = gs[0, :].subgridspec(
        3, 1,
        height_ratios=[1.0, 0.40, 1.0],
        hspace=0.0
    )
    ax_table_train = fig.add_subplot(top_gs[0, 0])
    ax_table_gap   = fig.add_subplot(top_gs[1, 0])
    ax_table_meta  = fig.add_subplot(top_gs[2, 0])
    ax_table_gap.axis("off")

    # row 1: loss
    ax_loss_left   = fig.add_subplot(gs[1, 0])
    ax_loss_right  = fig.add_subplot(gs[1, 1])

    # row 2: local standalone per-task F1
    ax_local_standalone_left  = fig.add_subplot(gs[2, 0])
    ax_local_standalone_right = fig.add_subplot(gs[2, 1])

    # row 3: fedavg client-local pre-aggregation
    ax_preagg_left  = fig.add_subplot(gs[3, 0])
    ax_preagg_right = fig.add_subplot(gs[3, 1])

    # row 4: fedavg global post-aggregation
    ax_postagg_left  = fig.add_subplot(gs[4, 0])
    ax_postagg_right = fig.add_subplot(gs[4, 1])

    # row 5: delta local standalone - fedavg global
    ax_delta_left   = fig.add_subplot(gs[5, 0])
    ax_delta_right  = fig.add_subplot(gs[5, 1])

    # row 6: pooled
    ax_pool_local   = fig.add_subplot(gs[6, 0])
    ax_pool_global  = fig.add_subplot(gs[6, 1])

    # headers
    fig.text(
        0.01, 0.985,
        build_pair_family_footer(row_left),
        ha="left", va="top", fontsize=11,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="0.70",
            alpha=0.95,
        ),
    )

    fig.text(
        0.5, 0.942,
        f"subset=[{subset_clients}] | family={family} | model={model_tag} | local_epochs={local_epochs} | n_seeds={n_seeds}",
        ha="center", va="top", fontsize=16, fontweight="semibold"
    )

    # top tables
    draw_pair_train_stats_table(
        ax_table_train,
        graph_id_left=graph_id_left,
        graph_id_right=graph_id_right,
        graph_meta_df=graph_meta_df,
        task_names=TASKS,
        fontsize=12,
    )

    draw_pair_metric_detail_table(
        ax_table_meta,
        pair_row=row_left,
        graph_id_left=graph_id_left,
        graph_id_right=graph_id_right,
        fontsize=10,
    )

    # -------------------------------------------------
    # Row 1: validation loss comparison
    # -------------------------------------------------
    local_loss_left = get_scalar_seed_stats(
        local_dfs_left, phase="val_epoch", split="val",
        metric_col="eval_loss", local_epochs=local_epochs
    )
    fed_pre_loss_left = get_scalar_seed_stats(
        fed_dfs_left, phase="val_epoch", split="val",
        metric_col="eval_loss", graph_id=graph_id_left, local_epochs=local_epochs
    )
    fed_global_loss_left = get_scalar_seed_stats(
        fed_dfs_left, phase="global_val_client", split="val",
        metric_col="eval_loss", graph_id=graph_id_left, local_epochs=local_epochs
    )

    plot_mean_std(ax_loss_left, local_loss_left, label="local standalone val", color="tab:blue")
    plot_mean_std(ax_loss_left, fed_pre_loss_left, label="fedavg client-local val (pre-aggregation)", color="tab:orange")
    plot_mean_std(ax_loss_left, fed_global_loss_left, label="fedavg global val on client (post-aggregation)", color="tab:green")
    ax_loss_left.set_title(f"Graph {graph_id_left}: validation loss comparison")
    ax_loss_left.set_xlabel("communication round / local epoch")
    ax_loss_left.set_ylabel("eval_loss")
    ax_loss_left.set_ylim(0, 3)
    ax_loss_left.grid(alpha=0.3)
    ax_loss_left.legend(fontsize=8)

    local_loss_right = get_scalar_seed_stats(
        local_dfs_right, phase="val_epoch", split="val",
        metric_col="eval_loss", local_epochs=local_epochs
    )
    fed_pre_loss_right = get_scalar_seed_stats(
        fed_dfs_right, phase="val_epoch", split="val",
        metric_col="eval_loss", graph_id=graph_id_right, local_epochs=local_epochs
    )
    fed_global_loss_right = get_scalar_seed_stats(
        fed_dfs_right, phase="global_val_client", split="val",
        metric_col="eval_loss", graph_id=graph_id_right, local_epochs=local_epochs
    )

    plot_mean_std(ax_loss_right, local_loss_right, label="local standalone val", color="tab:blue")
    plot_mean_std(ax_loss_right, fed_pre_loss_right, label="fedavg client-local val (pre-aggregation)", color="tab:orange")
    plot_mean_std(ax_loss_right, fed_global_loss_right, label="fedavg global val on client (post-aggregation)", color="tab:green")
    ax_loss_right.set_title(f"Graph {graph_id_right}: validation loss comparison")
    ax_loss_right.set_xlabel("communication round / local epoch")
    ax_loss_right.set_ylabel("eval_loss")
    ax_loss_right.set_ylim(0, 3)
    ax_loss_right.grid(alpha=0.3)
    ax_loss_right.legend(fontsize=8)

    # -------------------------------------------------
    # Row 2: local standalone client F1 per task
    # -------------------------------------------------
    has_any = False
    for task in TASKS:
        agg = get_task_seed_stats(
            local_dfs_left,
            phase="val_epoch_task",
            split="val",
            task=task,
            metric_col="positive_f1",
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_local_standalone_left, agg, label=task, color=TASK_COLORS[task])

    ax_local_standalone_left.set_title(f"Graph {graph_id_left}: local standalone on client (positive F1)")
    ax_local_standalone_left.set_xlabel("communication round / local epoch")
    ax_local_standalone_left.set_ylabel("positive_f1")
    ax_local_standalone_left.set_ylim(0, 1)
    ax_local_standalone_left.grid(alpha=0.3)
    if has_any:
        ax_local_standalone_left.legend(fontsize=8, ncol=2)

    has_any = False
    for task in TASKS:
        agg = get_task_seed_stats(
            local_dfs_right,
            phase="val_epoch_task",
            split="val",
            task=task,
            metric_col="positive_f1",
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_local_standalone_right, agg, label=task, color=TASK_COLORS[task])

    ax_local_standalone_right.set_title(f"Graph {graph_id_right}: local standalone on client (positive F1)")
    ax_local_standalone_right.set_xlabel("communication round / local epoch")
    ax_local_standalone_right.set_ylabel("positive_f1")
    ax_local_standalone_right.set_ylim(0, 1)
    ax_local_standalone_right.grid(alpha=0.3)
    if has_any:
        ax_local_standalone_right.legend(fontsize=8, ncol=2)

    # -------------------------------------------------
    # Row 3: FedAvg client-local on client (pre-aggregation)
    # -------------------------------------------------
    has_any = False
    for task in TASKS:
        agg = get_task_seed_stats(
            fed_dfs_left,
            phase="val_epoch_task",
            split="val",
            task=task,
            metric_col="positive_f1",
            graph_id=graph_id_left,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_preagg_left, agg, label=task, color=TASK_COLORS[task])

    ax_preagg_left.set_title(f"Graph {graph_id_left}: FedAvg client-local on client (pre-aggregation, positive F1)")
    ax_preagg_left.set_xlabel("communication round / local epoch")
    ax_preagg_left.set_ylabel("positive_f1")
    ax_preagg_left.set_ylim(0, 1)
    ax_preagg_left.grid(alpha=0.3)
    if has_any:
        ax_preagg_left.legend(fontsize=8, ncol=2)

    has_any = False
    for task in TASKS:
        agg = get_task_seed_stats(
            fed_dfs_right,
            phase="val_epoch_task",
            split="val",
            task=task,
            metric_col="positive_f1",
            graph_id=graph_id_right,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_preagg_right, agg, label=task, color=TASK_COLORS[task])

    ax_preagg_right.set_title(f"Graph {graph_id_right}: FedAvg client-local on client (pre-aggregation, positive F1)")
    ax_preagg_right.set_xlabel("communication round / local epoch")
    ax_preagg_right.set_ylabel("positive_f1")
    ax_preagg_right.set_ylim(0, 1)
    ax_preagg_right.grid(alpha=0.3)
    if has_any:
        ax_preagg_right.legend(fontsize=8, ncol=2)

    # -------------------------------------------------
    # Row 4: FedAvg global on client (post-aggregation)
    # -------------------------------------------------
    has_any = False
    for task in TASKS:
        agg = get_task_seed_stats(
            fed_dfs_left,
            phase="global_val_client_task",
            split="val",
            task=task,
            metric_col="positive_f1",
            graph_id=graph_id_left,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_postagg_left, agg, label=task, color=TASK_COLORS[task])

    ax_postagg_left.set_title(f"Graph {graph_id_left}: FedAvg global on client (post-aggregation, positive F1)")
    ax_postagg_left.set_xlabel("communication round / local epoch")
    ax_postagg_left.set_ylabel("positive_f1")
    ax_postagg_left.set_ylim(0, 1)
    ax_postagg_left.grid(alpha=0.3)
    if has_any:
        ax_postagg_left.legend(fontsize=8, ncol=2)

    has_any = False
    for task in TASKS:
        agg = get_task_seed_stats(
            fed_dfs_right,
            phase="global_val_client_task",
            split="val",
            task=task,
            metric_col="positive_f1",
            graph_id=graph_id_right,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_postagg_right, agg, label=task, color=TASK_COLORS[task])

    ax_postagg_right.set_title(f"Graph {graph_id_right}: FedAvg global on client (post-aggregation, positive F1)")
    ax_postagg_right.set_xlabel("communication round / local epoch")
    ax_postagg_right.set_ylabel("positive_f1")
    ax_postagg_right.set_ylim(0, 1)
    ax_postagg_right.grid(alpha=0.3)
    if has_any:
        ax_postagg_right.legend(fontsize=8, ncol=2)

    # -------------------------------------------------
    # Row 5: local standalone - FedAvg global delta
    # -------------------------------------------------
    has_any = False
    for task in TASKS:
        agg = get_delta_seed_stats(
            local_dfs_left,
            fed_dfs_left,
            phase_left="val_epoch_task",
            phase_right="global_val_client_task",
            split="val",
            metric_col="positive_f1",
            graph_id_right=graph_id_left,
            task=task,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_delta_left, agg, label=task, color=TASK_COLORS[task])

    ax_delta_left.axhline(0.0, linestyle="--", linewidth=1, color="black")
    ax_delta_left.set_title(f"Graph {graph_id_left}: delta = local standalone - FedAvg global")
    ax_delta_left.set_xlabel("communication round / local epoch")
    ax_delta_left.set_ylabel("positive_f1 delta")
    ax_delta_left.set_ylim(-1, 1)
    ax_delta_left.grid(alpha=0.3)
    if has_any:
        ax_delta_left.legend(fontsize=8, ncol=2)

    has_any = False
    for task in TASKS:
        agg = get_delta_seed_stats(
            local_dfs_right,
            fed_dfs_right,
            phase_left="val_epoch_task",
            phase_right="global_val_client_task",
            split="val",
            metric_col="positive_f1",
            graph_id_right=graph_id_right,
            task=task,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(ax_delta_right, agg, label=task, color=TASK_COLORS[task])

    ax_delta_right.axhline(0.0, linestyle="--", linewidth=1, color="black")
    ax_delta_right.set_title(f"Graph {graph_id_right}: delta = local standalone - FedAvg global")
    ax_delta_right.set_xlabel("communication round / local epoch")
    ax_delta_right.set_ylabel("positive_f1 delta")
    ax_delta_right.set_ylim(-1, 1)
    ax_delta_right.grid(alpha=0.3)
    if has_any:
        ax_delta_right.legend(fontsize=8, ncol=2)

    # -------------------------------------------------
    # Row 6: pooled comparisons
    # -------------------------------------------------
    has_any = False
    for task in TASKS:
        pooled_local = get_pair_pooled_task_f1_seed_stats(
            local_dfs_left, local_dfs_right,
            phase="val_epoch_task", split="val",
            task=task,
            graph_id_left=None, graph_id_right=None,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(
            ax_pool_local, pooled_local,
            label=task, color=TASK_COLORS[task],
            linestyle="-", alpha_fill=0.12, linewidth=2.0,
        )

    ax_pool_local.set_title("Pair pooled LOCAL per-task F1")
    ax_pool_local.set_xlabel("communication round / local epoch")
    ax_pool_local.set_ylabel("pooled per-task F1")
    ax_pool_local.set_ylim(0, 1)
    ax_pool_local.grid(alpha=0.3)
    if has_any:
        ax_pool_local.legend(fontsize=8, ncol=2)

    has_any = False
    for task in TASKS:
        pooled_global = get_pair_pooled_task_f1_seed_stats(
            fed_dfs_left, fed_dfs_right,
            phase="global_val_client_task", split="val",
            task=task,
            graph_id_left=graph_id_left, graph_id_right=graph_id_right,
            local_epochs=local_epochs,
        )
        has_any |= plot_mean_std(
            ax_pool_global, pooled_global,
            label=task, color=TASK_COLORS[task],
            linestyle="-", alpha_fill=0.12, linewidth=2.0,
        )

    ax_pool_global.set_title("Pair pooled FedAvg GLOBAL per-task F1")
    ax_pool_global.set_xlabel("communication round / local epoch")
    ax_pool_global.set_ylabel("pooled per-task F1")
    ax_pool_global.set_ylim(0, 1)
    ax_pool_global.grid(alpha=0.3)
    if has_any:
        ax_pool_global.legend(fontsize=8, ncol=2)

    fig.subplots_adjust(
        top=0.92,
        bottom=0.06,
        left=0.05,
        right=0.98,
        hspace=0.45,
        wspace=0.24,
    )

    if save_root is not None:
        out_dir = Path(save_root)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"pair_{subset_clients}_{model_tag}.pdf"
        fig.savefig(out_path, bbox_inches="tight")

    return fig
    
def draw_pair_metric_detail_table(ax, pair_row, graph_id_left, graph_id_right, fontsize=11):
    ax.axis("off")
    ax.set_anchor("N")

    rows = family_metric_rows(pair_row)

    if not rows:
        ax.text(0.5, 0.5, "No family metric rows available", ha="center", va="center", fontsize=fontsize)
        return

    row_labels = [r[0] for r in rows]
    cell_text = [[r[1], r[2], r[3]] for r in rows]

    table = ax.table(
        cellText=cell_text,
        rowLabels=row_labels,
        colLabels=[f"graph {graph_id_left}", "gap / pair", f"graph {graph_id_right}"],
        loc="center",
        cellLoc="center",
        rowLoc="center",
        bbox=[0, 0, 1, 1],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)

    n_rows = len(rows)
    row_scale = 1.10 if n_rows <= 6 else 0.98 if n_rows <= 8 else 0.88
    table.scale(1.03, row_scale)

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("0.75")
        if r == 0 or c == -1:
            cell.set_text_props(weight="bold")
        if r == 0:
            cell.set_facecolor("#f2f2f2")

# Plot helpers epoch 1 and epoch 3

In [14]:
def build_pair_pdf_from_log(
    *,
    selected_pairs_df,
    exp_log_df,
    graph_meta_df,
    out_pdf_path,
):
    pair_run_table = build_pair_run_table(selected_pairs_df, exp_log_df)

    with PdfPages(out_pdf_path) as pdf:
        for (subset_clients, model_tag), grp in pair_run_table.groupby(
            ["subset_clients", "model_tag"], sort=False
        ):
            if len(grp) != 2:
                print(
                    f"Skipping {subset_clients} / {model_tag}: "
                    f"expected 2 graphs, got {len(grp)}"
                )
                continue

            fig = make_pair_overview_figure(
                pair_group_df=grp,
                graph_meta_df=graph_meta_df,
                save_root=None,
            )
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print(f"saved -> {out_pdf_path}")

## Runner

In [15]:
build_pair_pdf_from_log(
    selected_pairs_df=selected_pairs,
    exp_log_df=exp_log_epoch1,
    graph_meta_df=test_gen,
    out_pdf_path=Path(f"./{EXPERIMENT_LOG_FOLDER}/fedavg_pair_overview_epoch1.pdf"),
)

build_pair_pdf_from_log(
    selected_pairs_df=selected_pairs,
    exp_log_df=exp_log_epoch3,
    graph_meta_df=test_gen,
    out_pdf_path=Path(f"./{EXPERIMENT_LOG_FOLDER}/fedavg_pair_overview_epoch3.pdf"),
)

build_pair_pdf_from_log(
    selected_pairs_df=selected_pairs,
    exp_log_df=exp_log_epoch5,
    graph_meta_df=test_gen,
    out_pdf_path=Path(f"./{EXPERIMENT_LOG_FOLDER}/fedavg_pair_overview_epoch5.pdf"),
)

build_pair_pdf_from_log(
    selected_pairs_df=selected_pairs,
    exp_log_df=exp_log_epoch10,
    graph_meta_df=test_gen,
    out_pdf_path=Path(f"./{EXPERIMENT_LOG_FOLDER}/fedavg_pair_overview_epoch10.pdf"),
)

saved -> pairwise_selection_experiment/fedavg_pair_overview_epoch1.pdf
saved -> pairwise_selection_experiment/fedavg_pair_overview_epoch3.pdf
saved -> pairwise_selection_experiment/fedavg_pair_overview_epoch5.pdf
saved -> pairwise_selection_experiment/fedavg_pair_overview_epoch10.pdf
